# Geographic scope classification — eval notebook

Tests the two-pass geographic scope pipeline against the `/nli` endpoint.

**Decision logic** (implemented here, not in the service):

1. **Pass 1 — top-level scope**: call `/nli` with the three scope hypotheses (national / regional / city), no threshold.  
   Pick the highest-scoring hypothesis → scope verdict.  
   If the highest score < `scope_threshold` → scope = `unknown`.
2. **Pass 2 — city disambiguation** (run when `city_candidates` is non-empty):  
   For each candidate city call `/nli` with hypothesis `"Este artículo trata principalmente sobre movilidad urbana en {city}."`.  
   Rank candidates by score; the top one is the resolved city.

Test cases live in `eval/fixtures/nli-news-scope.json`.

In [9]:
import os, json, requests, yaml
from pathlib import Path

NLP_BASE_URL = os.environ.get('NLP_BASE_URL', 'https://wiig.dia.fi.upm.es/nlp').rstrip('/')
ENDPOINT     = f'{NLP_BASE_URL}/nli'
HEADERS      = {'Content-Type': 'application/json', 'X-API-Key': f'{os.environ.get("NLP_API_KEY")}'}
TIMEOUT      = 120

cfg = yaml.safe_load(Path('../config/topics.yaml').read_text(encoding='utf-8'))
SCOPE_HYPOTHESES = cfg['scope_hypotheses']   # dict: national / regional / city
SCOPE_THRESHOLD  = cfg['scope_threshold']

CASES = json.loads(Path('fixtures/nli-news-scope.json').read_text(encoding='utf-8'))
print(f'{len(CASES)} cases -> {ENDPOINT}')
print(f'scope_threshold={SCOPE_THRESHOLD}')
print('scope hypotheses:', list(SCOPE_HYPOTHESES.keys()))

6 cases -> https://wiig.dia.fi.upm.es/b4c_nlp/nli
scope_threshold=0.35
scope hypotheses: ['national', 'regional', 'city']


In [10]:
def build_premise(inp):
    headline = (inp.get('headline') or '').strip()
    text     = (inp.get('text') or '').strip()
    return f'{headline}. {text}' if headline else text


def call_nli(premise, hypotheses):
    body = {'text': premise, 'hypotheses': hypotheses}
    r = requests.post(ENDPOINT, json=body, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json().get('scores', [])


def classify_scope(inp):
    """Return (scope, city, pass1_scores, pass2_city_scores)."""
    premise    = build_premise(inp)
    candidates = inp.get('city_candidates', [])

    # Pass 1 — top-level scope
    scope_keys  = list(SCOPE_HYPOTHESES.keys())
    scope_hyps  = [SCOPE_HYPOTHESES[k].strip() for k in scope_keys]
    p1_raw      = call_nli(premise, scope_hyps)
    # map back to keys by position
    p1_scores   = [
        {'scope': scope_keys[i], 'hypothesis': item['hypothesis'], 'score': item['score']}
        for i, item in enumerate(p1_raw)
    ]
    best        = max(p1_scores, key=lambda x: x['score'])
    scope       = best['scope'] if best['score'] >= SCOPE_THRESHOLD else 'unknown'

    # Pass 2 — city disambiguation
    city_scores = []
    resolved_city = None
    if candidates:
        city_hyps = [
            f'Este artículo trata principalmente sobre movilidad urbana en {c}.'
            for c in candidates
        ]
        p2_raw = call_nli(premise, city_hyps)
        city_scores = [
            {'city': candidates[i], 'score': item['score']}
            for i, item in enumerate(p2_raw)
        ]
        city_scores.sort(key=lambda x: x['score'], reverse=True)
        resolved_city = city_scores[0]['city'] if city_scores else None

    return scope, resolved_city, p1_scores, city_scores

In [11]:
results = []
for case in CASES:
    print(f"── {case['id']}: {case['description']}")
    try:
        scope, resolved_city, p1_scores, city_scores = classify_scope(case['input'])
    except Exception as exc:
        print('   ERROR:', exc)
        print()
        results.append((case['id'], False))
        continue

    # Pass 1 render
    print(f'  Pass 1 — scope scores  (threshold={SCOPE_THRESHOLD})')
    for item in sorted(p1_scores, key=lambda x: x['score'], reverse=True):
        marker = ' <-- best' if item['scope'] == (scope if scope != 'unknown' else '') else ''
        print(f'    {item["scope"]:10s}  score={item["score"]:.3f}{marker}')
    print(f'  Scope verdict: {scope}')

    # Pass 2 render
    if city_scores:
        print(f'  Pass 2 — city disambiguation')
        for item in city_scores:
            marker = ' <-- top' if item['city'] == resolved_city else ''
            print(f'    {item["city"]:20s}  score={item["score"]:.3f}{marker}')
        print(f'  Resolved city: {resolved_city}')

    # Score
    exp         = case['expected']
    scope_ok    = scope == exp['scope']
    city_ok     = resolved_city == exp.get('city')  # None == None if no candidates
    passed      = scope_ok and city_ok
    icon        = '✅' if passed else '❌'
    mismatch    = []
    if not scope_ok:
        mismatch.append(f'expected scope={exp["scope"]!r}')
    if not city_ok:
        mismatch.append(f'expected city={exp.get("city")!r}')
    note = '  ' + ', '.join(mismatch) if mismatch else ''
    print(f'  {icon} case {case["id"]}{note}')
    print()
    results.append((case['id'], passed))

── scope-001: National cycling safety law — national scope, no city candidates
  Pass 1 — scope scores  (threshold=0.35)
    regional    score=0.192
    city        score=0.153
    national    score=0.001
  Scope verdict: unknown
  ❌ case scope-001  expected scope='national'

── scope-002: Cataluña regional sustainable mobility plan — regional scope, no city candidates
  Pass 1 — scope scores  (threshold=0.35)
    regional    score=0.636 <-- best
    national    score=0.435
    city        score=0.139
  Scope verdict: regional
  ✅ case scope-002

── scope-003: Madrid bike lane expansion — city scope, candidates Madrid and Barcelona
  Pass 1 — scope scores  (threshold=0.35)
    city        score=0.410 <-- best
    regional    score=0.042
    national    score=0.001
  Scope verdict: city
  Pass 2 — city disambiguation
    Madrid                score=0.050 <-- top
    Barcelona             score=0.001
  Resolved city: Madrid
  ✅ case scope-003

── scope-004: Barcelona Superilla pedestrian

In [12]:
passed_n = sum(1 for _, p in results if p)
print('SCORECARD · /nli  (geographic scope classification)')
print(f'  Cases passing: {passed_n}/{len(results)}')
for cid, p in results:
    print(f"   {'✅' if p else '❌'} {cid}")

SCORECARD · /nli  (geographic scope classification)
  Cases passing: 2/6
   ❌ scope-001
   ✅ scope-002
   ✅ scope-003
   ❌ scope-004
   ❌ scope-005
   ❌ scope-006
